In [8]:
import re
import pandas as pd
from collections import defaultdict
import pdfplumber

In [9]:
text_pdf = r"/Users/thenning/Downloads/Caltech (0-1,0-0 SCIAC) -vs- UC Santa Cruz (1-0,0-0 C2C) - ACFrOgDR4i2_CYN1mMnSaY03YB4kJcVOqlp8Gemrcplp27ygYTHiffiq3IkOFZIVVD6ZnfRIuPsKL_Oe8iczD8X_VeH2P5klKde-8c2DdPxFcKiZX68fMtmrfeRGB6CtqfF-L40f617O59x3Rq0j.pdf"

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text_pages = []
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                t = page.extract_text() or ""
                text_pages.append(t)
        text = "\n".join(text_pages)
        if text.strip():
            return text
    except Exception as e:
        print(f"[extract] pdfplumber error: {e}")

    # fallback
    try:
        import PyPDF2
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                t = page.extract_text() or ""
                text_pages.append(t)
        text = "\n".join(text_pages)
        return text
    except Exception as e:
        print(f"[extract] PyPDF2 error: {e}")
        return ""
    
def normalize_text(t: str) -> str:
    # Keep newlines; just normalize unicode dashes to '-' to stabilize regex.
    t = t.replace("\u2013", "-").replace("\u2014", "-")
    return t

def count_matches(text: str, pattern: str, flags=re.I|re.S) -> int:
    return len(re.findall(pattern, text, flags))


In [21]:
raw_text = extract_text_from_pdf(text_pdf)
normalized_text = normalize_text(raw_text)
normalized_text

"Caltech (0-1,0-0 SCIAC) -vs- UC Santa Cruz (1-0,0-0 C2C)\n11/08/24 at Santa Cruz, CA\nDate:11/08/24 ScoreByPeriod 1 2 Total\nTime:4:00PM\nCaltech 23 23 46\nAttendance:100\nSite:SantaCruz,CA UCSantaCruz 41 55 96\nNotes:2024-25Men'sBasketball.Round:0.CaltechvsUCSanta\nCruz.PlayedatKaiser\nCaltech 46\n# Player GS MIN FG 3PT FT ORB-DRB REB PF A TO BLK STL PTS\n04 NoahHicks * 32 5-14 0-6 1-1 0-5 5 1 1 0 0 1 11\n30 RyanHu * 15 2-4 1-3 2-2 0-2 2 3 2 2 0 0 7\n22 ChaseWilliamson * 30 1-10 0-3 4-6 3-3 6 2 4 3 0 0 6\n34 EvanPortnoi * 22 1-3 1-2 2-2 1-0 1 1 1 0 0 0 5\n23 RileyWood * 17 1-2 0-0 0-2 1-2 3 1 0 1 0 1 2\n40 StuartFlorescu 22 1-6 0-0 2-4 1-3 4 2 0 3 1 0 4\n10 StephenLo 6 1-1 1-1 1-2 0-0 0 0 0 3 0 0 4\n00 JoshBalami 13 1-6 0-1 1-2 1-2 3 1 0 3 0 0 3\n24 JamesDowns 14 0-2 0-1 2-2 1-0 1 3 0 0 0 0 2\n05 PranitGunjal 8 1-1 0-0 0-0 0-0 0 1 0 0 0 0 2\n03 DagemawiGetachew 7 0-0 0-0 0-0 0-1 1 0 0 3 0 0 0\n21 AndrewKoclanes 6 0-1 0-1 0-0 0-0 0 0 0 0 0 0 0\n11 RobertJackson 4 0-1 0-1 0-0 0-0 0 1 0

In [42]:
import re
from pathlib import Path

# -------------------------------
# 1. PDF extraction
# -------------------------------
def extract_text_from_pdf(pdf_path: str) -> str:
    text_pages = []
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text() or ""
                text_pages.append(page_text)
        text = "\n".join(text_pages)
        if text.strip():
            return text
    except Exception as e:
        print(f"[extract] pdfplumber error: {e}")

    # fallback: PyPDF2
    try:
        import PyPDF2
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text() or ""
                text_pages.append(page_text)
        return "\n".join(text_pages)
    except Exception as e:
        print(f"[extract] PyPDF2 error: {e}")
        return ""

# -------------------------------
# 2. Normalization helpers
# -------------------------------
def normalize_text(t: str) -> str:
    # Replace unicode dashes and collapse huge spaces
    t = t.replace("\u2013", "-").replace("\u2014", "-")
    return t

def normalize_pbp_tokens(text: str) -> str:
    """
    Fix missing spaces in play-by-play lines such as:
    SUBINby → SUB IN by
    GOOD3PTRby → GOOD 3PTR by
    MISSJUMPERby → MISS JUMPER by
    """
    # Handle substitutions
    text = re.sub(r"SUBINby", "SUB IN by", text, flags=re.I)
    text = re.sub(r"SUBOUTby", "SUB OUT by", text, flags=re.I)

    # Handle scoring and shot events (both GOOD and MISS)
    text = re.sub(r"GOOD(\w+)by", r"GOOD \1 by", text, flags=re.I)
    text = re.sub(r"MISS(\w+)by", r"MISS \1 by", text, flags=re.I)

    # Handle rebounds (sometimes 'REBOUNDEDFby' or 'REBOUNDBY')
    text = re.sub(r"REBOUNDEDF?by", "REBOUNDED by", text, flags=re.I)

    return text

# -------------------------------
# 3. Diagnostics / probes
# -------------------------------
def count_matches(text: str, pattern: str, flags=re.I|re.S) -> int:
    return len(re.findall(pattern, text, flags))

def show_context_lines(text: str, pattern: str, which=1, radius=15, flags=re.I):
    lines = text.splitlines()
    idx_map = []
    pos = 0
    for i, ln in enumerate(lines):
        idx_map.append((i, pos, pos + len(ln)))
        pos += len(ln) + 1
    matches = list(re.finditer(pattern, text, flags))
    if not matches:
        print(f"[context] No matches for: {pattern!r}")
        return
    m = matches[min(which-1, len(matches)-1)]
    start = m.start()
    line_idx = None
    for i, a, b in idx_map:
        if a <= start <= b:
            line_idx = i
            break
    lo = max(0, line_idx - radius)
    hi = min(len(lines), line_idx + radius + 1)
    print(f"\n[context] Showing lines {lo}..{hi} around match of {pattern!r}:\n")
    for j in range(lo, hi):
        prefix = ">> " if j == line_idx else "   "
        print(prefix + lines[j])

# -------------------------------
# 4. Run step
# -------------------------------
if __name__ == "__main__":
    PDF_PATH = text_pdf  # <--- your PDF file

    # Extract + normalize
    raw = extract_text_from_pdf(PDF_PATH)
    if not raw.strip():
        raise SystemExit("[fatal] Could not extract any text from the PDF.")
    text = normalize_text(raw)
    text = normalize_pbp_tokens(text)

    # # Save for inspection
    # txt_out = Path(PDF_PATH).with_suffix(".txt")
    # txt_out.write_text(text, encoding="utf-8")
    # print(f"[ok] Wrote normalized text to: {txt_out}")

    # Probe counts
    probes = {
        "# Player tables": r"#\s*Player",
        "Team Summary blocks": r"Team\s*Summary",
        "Totals - 200 markers": r"Totals\s*-\s*200",
        "Caltech mention": r"\bCaltech\b",
        "Play-by-play H1": r"1st\s*Half\s*Play\s*By\s*Play",
        "Play-by-play H2": r"2nd\s*Half\s*Play\s*By\s*Play",
        "Any 'SUB IN by'": r"SUB\s*IN\s*by",
        "Any 'SUB OUT by'": r"SUB\s*OUT\s*by",
        "Any 'GOOD 3PTR'": r"GOOD\s*3PTR\s*by",
        "Any 'MISS 3PTR'": r"MISS\s*3PTR\s*by",
        "Any 'GOOD FT'": r"GOOD\s*FT\s*by",
        "Any 'MISS FT'": r"MISS\s*FT\s*by",
        "Any 'GOOD LAYUP'": r"GOOD\s*LAYUP\s*by",
        "Any 'MISS Layup'": r"MISS\s*LAYUP\s*by",
        "Any 'GOOD JUMPER'": r"GOOD\s*JUMPER\s*by",
        "Any 'MISS JUMPER'": r"MISS\s*JUMPER\s*by",
        "Any 'GOOD TIPIN'": r"GOOD\s*TIPIN\s*by",
        "Any 'MISS TIPIN'": r"MISS\s*TIPIN\s*by",
        "Any 'GOOD DUNK'": r"GOOD\s*DUNK\s*by",
        "Any 'MISS DUNK'": r"MISS\s*DUNK\s*by",
        "Any 'TURNOVER'": r"TURNOVER\s*by",
        "Any 'OFF REBOUND'": r"REBOUNDOFF\s*by",
        "Any 'DEF REBOUND'": r"REBOUNDDEF\s*by",
        "Any 'BLOCK'": r"BLOCK\s*by",
        "Any 'STEAL'": r"STEAL\s*by",
        "Any 'FOUL": r"FOUL\s*by",
        "Any 'ASSIST'": r"ASSIST\s*by",






    }

    print("\n=== Probe counts ===")
    for label, pat in probes.items():
        c = count_matches(text, pat)
        print(f"{label:22s}: {c}")

    # Peek context around player table and PBP
    show_context_lines(text, r"#\s*Player", which=1, radius=15)
    show_context_lines(text, r"1st\s*Half\s*Play\s*By\s*Play", which=1, radius=20)
    show_context_lines(text, r"2nd\s*Half\s*Play\s*By\s*Play", which=1, radius=20)



=== Probe counts ===
# Player tables       : 6
Team Summary blocks   : 2
Totals - 200 markers  : 2
Caltech mention       : 7
Play-by-play H1       : 1
Play-by-play H2       : 1
Any 'SUB IN by'       : 76
Any 'SUB OUT by'      : 76
Any 'GOOD 3PTR'       : 16
Any 'MISS 3PTR'       : 47
Any 'GOOD FT'         : 28
Any 'MISS FT'         : 13
Any 'GOOD LAYUP'      : 23
Any 'MISS Layup'      : 17
Any 'GOOD JUMPER'     : 6
Any 'MISS JUMPER'     : 11
Any 'GOOD TIPIN'      : 1
Any 'MISS TIPIN'      : 0
Any 'GOOD DUNK'       : 3
Any 'MISS DUNK'       : 0
Any 'TURNOVER'        : 31
Any 'OFF REBOUND'     : 21
Any 'DEF REBOUND'     : 50
Any 'BLOCK'           : 3
Any 'STEAL'           : 12
Any 'FOUL             : 39
Any 'ASSIST'          : 33

[context] Showing lines 0..26 around match of '#\\s*Player':

   Caltech (0-1,0-0 SCIAC) -vs- UC Santa Cruz (1-0,0-0 C2C)
   11/08/24 at Santa Cruz, CA
   Date:11/08/24 ScoreByPeriod 1 2 Total
   Time:4:00PM
   Caltech 23 23 46
   Attendance:100
   Site:SantaC

In [44]:
import re
import pandas as pd

def extract_team_totals(text):
    """
    Parse the team 'Totals' row for each box-score table.
    Returns a DataFrame with columns:
    team, fgm, fga, 3pm, 3pa, ftm, fta, pts, off_reb, def_reb, reb, ast, stl, blk, to, pf
    (some columns may be missing if not in the PDF)
    """
    blocks = re.findall(r"#\s*Player.*?Team\s*Summary", text, flags=re.S | re.I)
    teams = []
    for i, block in enumerate(blocks, start=1):
        m = re.search(r"Totals\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)\s+([\d\-]+)", block)
        if m:
            nums = [int(x) if x.isdigit() else 0 for x in m.groups()]
            row = dict(zip(
                ["fgm","fga","3pm","3pa","ftm","fta","pts","off_reb","def_reb","reb","ast","stl","blk","pf"],
                nums
            ))
            row["team_index"] = i
            teams.append(row)
        else:
            print(f"[warn] Couldn't find Totals line for team {i}")
    return pd.DataFrame(teams)


def test_team_totals(text, expected=None, tolerance=0):
    """
    Compare totals across teams. Optionally pass in 'expected' dict
    like {'Caltech': {'pts': 68, 'reb': 33}} to assert against known values.
    """
    df = extract_team_totals(text)
    print("\n=== Extracted Team Totals ===")
    print(df)

    if expected:
        for team, vals in expected.items():
            if team not in df.index:
                continue
            for stat, val in vals.items():
                if abs(df.loc[team, stat] - val) > tolerance:
                    raise AssertionError(f"{team} {stat} mismatch: {df.loc[team, stat]} vs {val}")

    # quick symmetry check
    if len(df) == 2:
        print("\nDifference in totals (Team1 - Team2):")
        print((df.iloc[0] - df.iloc[1])[["pts","reb","ast","stl","blk","pf"]])
    return df


# After you have `text = normalize_pbp_tokens(raw_text)`
totals_df = test_team_totals(text)


[warn] Couldn't find Totals line for team 1
[warn] Couldn't find Totals line for team 2

=== Extracted Team Totals ===
Empty DataFrame
Columns: []
Index: []


In [45]:
for line in text.splitlines():
    if re.search(r"Totals", line, flags=re.I):
        print(line)


Totals - 200 14-52 3-19 15-23 9-19 28 16 8 21 1 2 46
Totals - 200 35-72 13-44 13-18 12-31 43 23 25 10 2 10 96
Totals 100 7-26 1-8 8-14 4-11 15 6 5 9 1 2 23
Totals 100 14-37 7-26 6-8 5-17 22 14 9 5 0 3 41
Totals 100 7-26 2-11 7-9 5-8 13 10 3 12 0 0 23
Totals 100 21-35 6-18 7-10 7-14 21 9 16 5 2 7 55


In [47]:
import re
import pandas as pd

def parse_totals_line(line):
    # normalize
    line = line.strip()
    line = re.sub(r"(\d+)-(\d+)", r"\1 \2", line)
    nums = re.findall(r"\d+", line)
    if len(nums) < 10:
        return None
    # Convert to ints
    nums = list(map(int, nums))
    # Layout (variable lengths handled)
    keys = ["min","fgm","fga","3pm","3pa","ftm","fta","pts","off_reb","def_reb",
            "reb","ast","stl","blk","to","pf"]
    out = dict(zip(keys[:len(nums)], nums))
    out["raw"] = line
    return out


def extract_all_totals(text):
    lines = [ln for ln in text.splitlines() if re.search(r"Totals", ln, flags=re.I)]
    rows = [parse_totals_line(ln) for ln in lines if parse_totals_line(ln)]
    df = pd.DataFrame(rows)
    df["is_half"] = df["min"] == 100
    df["team_index"] = (df.index % 2) + 1  # assume alternating teams
    return df


def validate_half_vs_full(df, tolerance=0):
    results = []
    grouped = df.groupby("team_index")
    for team_idx, g in grouped:
        full = g[g["min"] == 200].iloc[0] if any(g["min"] == 200) else None
        halves = g[g["min"] == 100]
        if full is not None and len(halves) == 2:
            for col in ["fgm","fga","3pm","3pa","ftm","fta","pts","reb","ast","stl","blk","pf"]:
                half_sum = halves[col].sum()
                diff = full[col] - half_sum
                results.append({
                    "team": team_idx,
                    "stat": col,
                    "full": int(full[col]),
                    "halves_sum": int(half_sum),
                    "diff": int(diff),
                    "ok": abs(diff) <= tolerance
                })
    res_df = pd.DataFrame(results)
    print("\n=== Half vs Full Totals Consistency ===")
    print(res_df.pivot(index="stat", columns="team", values="diff"))
    mism = res_df[~res_df["ok"]]
    if not mism.empty:
        print("\n❌ Mismatches detected:")
        print(mism)
    else:
        print("\n✅ All half totals add up correctly to full totals.")
    return res_df

df_totals = extract_all_totals(text)
validate_half_vs_full(df_totals)



=== Half vs Full Totals Consistency ===
team  1  2
stat      
3pa   0  0
3pm   0  0
ast   0  0
blk   0  0
fga   0  0
fgm   0  0
fta   0  0
ftm   0  0
pf    0  0
pts   0  0
reb   0  0
stl   0  0

✅ All half totals add up correctly to full totals.


,team,stat,full,halves_sum,diff,ok
0,1,fgm,14,14,0,True
1,1,fga,52,52,0,True
2,1,3pm,3,3,0,True
3,1,3pa,19,19,0,True
4,1,ftm,15,15,0,True
5,1,fta,23,23,0,True
6,1,pts,9,9,0,True
7,1,reb,16,16,0,True
8,1,ast,8,8,0,True
9,1,stl,21,21,0,True


# Email Notification if fail occurs

In [50]:
!pip install MIMEMultipart

ERROR: Could not find a version that satisfies the requirement MIMEMultipart (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for MIMEMultipart


In [54]:
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import smtplib


def send_email_notification(subject, body, to_email, from_email, smtp_server, smtp_port=587, username=None, password=None):
    """
    Send an email alert using an SMTP server (e.g., Gmail, Caltech relay).
    - subject, body: message content
    - to_email: recipient string (or list)
    - from_email: sender address
    - smtp_server: e.g. 'smtp.gmail.com' or 'smtp.caltech.edu'
    - username/password: optional (for Gmail etc.)
    """
    msg = MIMEMultipart()
    msg["From"] = from_email
    msg["To"] = to_email if isinstance(to_email, str) else ", ".join(to_email)
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    try:
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.starttls()
            if username and password:
                server.login(username, password)
            server.send_message(msg)
        print(f"[email] Notification sent to {to_email}")
    except Exception as e:
        print(f"[email error] {e}")


email_config = {
    "to_email": "tlhennning@gmail.com",
    "from_email": "thenning@caltech.edu",
    "smtp_server": "smtp.gmail.com",
    "smtp_port": 587,
    "username": "thenning@caltech.edu",
    "password": "Batmoney15!!"
}

